# Pa-alpha rotation curve from NIRCam WFSS, one source

Fits a disk velocity field to the R and C dispersion directions of a single
source and reports a posterior, convergence gates and a rotation-vs-no-rotation
test. Set `SOURCE` below and run top to bottom.

The two directions are never co-added: they are perpendicular projections of
the same velocity field, and DINGO's loss is the difference between them after
each is rectified back to the direct-image frame.

**Inputs** `data/ID<id>/` must contain `config_kinematics.yaml`, `R.fits`,
`C.fits`, `image.fits`, and `results/ID<id>/` a pysersic fit for the
inclination. **Outputs** go to `results/ID<id>/`.

## Configuration

In [ ]:
SOURCE  = 16887        # EDR ID
DEVICE  = 'cuda'       # 'cuda' or 'cpu'
N_TEETH = 6            # staged fits run from the best N grid minima

RESUME  = False       # True continues this notebook's own chain if one exists

# None = take the value from the config's `mcmc:` block.
NSTEPS = None
NBURN  = None

In [ ]:
import os, sys
import numpy as np
import matplotlib.pyplot as plt

REPO = os.environ.get('DINGO_REPO', os.getcwd())
os.environ.setdefault('GRISM_CAL_DIR', os.path.join(os.path.dirname(REPO), 'grism_cal'))
sys.path.insert(0, REPO)

from dingo import fitting, plot, startpoint, utils

DATA = os.path.join(REPO, 'data', f'ID{SOURCE}')
OUT  = os.path.join(REPO, 'results', f'ID{SOURCE}')
os.makedirs(OUT, exist_ok=True)
CONFIG = os.path.join(DATA, 'config_kinematics.yaml')
out = lambda name: os.path.join(OUT, f'ID{SOURCE}_{name}')

# config `path:` entries are relative to data/
os.chdir(os.path.join(REPO, 'data'))

for f in ('config_kinematics.yaml', 'R.fits', 'C.fits', 'image.fits'):
    p = os.path.join(DATA, f)
    print(f'{"OK " if os.path.exists(p) else "MISSING"}  {f}')

## Inclination

The grism measures `V*sin(i)`, so the amplitude is degenerate with inclination
(corr ~ -0.99) unless `i` comes from imaging. It is pinned before Adam runs, so
the MAP, the frozen noise model and the posterior are all conditional on the
same geometry.

In [ ]:
sersic = np.load(out('sersic_F444W_pysersic_fixedinc.npz'), allow_pickle=True)
INC_RAD   = float(np.median(sersic['inc']))
INC_SIGMA = float(np.std(sersic['inc']))
print(f'i = {np.degrees(INC_RAD):.2f} +- {np.degrees(INC_SIGMA):.2f} deg '
      f'  (q = {np.median(sersic["q"]):.4f}, n = {np.median(sersic["n"]):.2f})')

## 1. Starting point

The R-C objective is periodic in the velocity amplitude: a shift of one
dispersion pixel re-registers R against C onto the neighbouring pixel for
nearly the same residual, so the loss carries a row of near-degenerate minima.
A single-start optimiser falls into whichever tooth it began in and reports
convergence from there, and every downstream diagnostic then looks healthy.

`find_start` scans `(amplitude, theta_v)` with forward evaluations only, then
runs the staged fit from each distinct minimum and keeps the lowest loss.
`theta_v` is scanned because `(V, theta)` and `(-V, theta+pi)` are the same
field -- which is what makes a per-source sign table unnecessary.

In [ ]:
build = lambda path: fitting.fix_inclination(
    fitting.KinematicsFitter(config_path=path, device=DEVICE), INC_RAD)

params, loss, trials = startpoint.find_start(
    CONFIG, device=DEVICE, n_keep=N_TEETH, make_fitter=build)

amp_key = startpoint.amplitude_name(build(CONFIG).config)
print(f'{"start":>8} {"final loss":>12} {amp_key:>10}')
for t in sorted(trials, key=lambda t: t['loss']):
    print(f'{t["start"][0]:8.1f} {t["loss"]:12.6f} {t["params"][amp_key]:10.2f}')

spread = max(t['loss'] for t in trials) - loss
print(f'\nbest loss {loss:.6f}; {len(trials)} teeth spanning {spread:.6f}')

## 2. Posterior

The fitter is rebuilt at the winning starting point, so `setup_noise_model`
freezes the mask and per-pixel variance there rather than at an arbitrary tooth.

**This cell takes hours.** Measured on ID16887: ~1.4 s per 32-walker step, so
the config's 10000 steps is ~4 h. The cost is `xy_iters: 40` in the config --
each likelihood call re-solves the registration offsets 40 times.

It checkpoints every 500 steps, so an interrupted run loses at most that much:
set `RESUME = True` and re-run to continue the same chain. Convergence arrives
well before the chain ends -- ID16887 passed every `n/tau > 50` gate at 3111
post-burn steps -- so a run stopped early is usable if section 3 passes.

In [ ]:
fit = build(CONFIG)
fit.set_params_dict({f'velocity.{k}': v for k, v in params.items()
                     if f'velocity.{k}' in fit._mcmc_stage_cfg()})
fit.fit_all()
fit.setup_noise_model()

# This chain is initialised from the scan winner, so it is NOT the same chain as
# one started from the config's single Adam point -- resuming across that would
# concatenate two posteriors. mcmc_fingerprint() guards priors and likelihood
# but cannot see the starting point, so give this pipeline its own checkpoint
# and leave any written by the old notebook untouched.
mcmc = fit.config.setdefault('mcmc', {})
mcmc['checkpoint'] = out('chain_scan_checkpoint.npz')
mcmc['resume'] = RESUME

kw = {k: v for k, v in (('nsteps', NSTEPS), ('nburn', NBURN)) if v is not None}
fit.fit_MCMC(**kw)
print(f'mean acceptance = {np.mean(fit.mcmc_sampler.acceptance_fraction):.3f}')

## 3. Gates

Acceptance outside 0.15-0.25, any `n/tau < 50`, or a split-half shift above 0.2
means the widths below are not quotable.

In [ ]:
d = fit.get_mcmc_diagnostics()
acc = d['acceptance_fraction'][0]
print(f"{d['n_steps']} steps after {d['n_burn']} burn   acceptance {acc:.3f}")
print(f"fraction of -inf evaluations: {d['frac_neg_inf']:.4f}\n")
print(f'{"parameter":<22}{"tau":>8}{"n/tau":>9}{"split shift":>13}  flag')
for n in fit.mcmc_param_names:
    ntau, shift = d['n_steps_over_tau'][n], d['split_half_shift'][n]
    flag = ', '.join(f for f, bad in (('n/tau<50', ntau <= 50),
                                      ('drifting', shift > 0.2)) if bad)
    print(f'{n:<22}{d["tau"][n]:>8.1f}{ntau:>9.1f}{shift:>13.3f}  {flag}')

worst = min(d['n_steps_over_tau'].values())
gates = {'acceptance 0.15-0.25': 0.15 <= acc <= 0.25, 'worst n/tau > 50': worst > 50}
print('\n' + '\n'.join(f'{"PASS" if ok else "FAIL"}  {k}' for k, ok in gates.items()))

## 4. Does it rotate?

`compare_null_model` re-optimises a `V = 0` model on the same frozen noise model
and multi-starts its offsets, because a systemic shift lowers the R-C residual
just as a velocity gradient does. A negative `dlnL` means the data prefer no
rotation and the amplitude is an upper limit, not a measurement.

In [ ]:
report = fit.fit_report(strict=False)
fit.set_params_to_mcmc('map')
null = fit.compare_null_model()
np.savez_compressed(out('null_model_comparison.npz'),
                    **{k: v for k, v in null.items() if not isinstance(v, dict)})

print(f"dlnL (rot - null) = {null['dlnL']:+.2f}   Wilks p = {null['p_wilks']:.2e} "
      f"(~{null['sigma_equiv']:.1f} sigma)   dBIC = {null['dBIC']:+.1f}")

# Statistical significance is necessary and nowhere near sufficient. ID16887
# returns dlnL +23.8 at 5.97 sigma for V*sin(i) = 8.4 km/s -- an eighth of the
# 68.2 km/s that one dispersion pixel subtends -- with R_v railed on its lower
# bound and stage 2 unconverged. That is a fit to noise that the Wilks test is
# happy to call significant, so the physical and convergence checks carry equal
# weight here.
amp_map = abs(fit.get_params()[0][amp_key])
V_LOS = amp_map if amp_key == 'V_sini' else amp_map*np.sin(INC_RAD)

# Resolution floor from the forward model, not a nominal dispersion, and with
# the module read from the config -- the wrong module rectifies the wrong pixels
# and still returns a plausible number.
summary = fit.config['summary']
KMS_PER_PIX = utils.kms_per_pixel(summary['lambda_rest']*(1 + summary['z']),
                                  summary['filter'],
                                  fit.config['image']['R']['module'])

checks = {
    'dlnL > 0':                       null['dlnL'] > 0,
    'dBIC > 0':                       null['dBIC'] > 0,
    'Wilks p < 1.35e-3 (3 sigma)':    null['p_wilks'] < 1.35e-3,
    f'V*sin(i) > 1 pixel ({KMS_PER_PIX:.1f} km/s)': V_LOS > KMS_PER_PIX,
    'fit raised no warnings':         report['ok'],
}
print()
for k, ok in checks.items():
    print(f'{"PASS" if ok else "FAIL"}  {k}')
for w in report['problems']:
    print(f'      ! {w}')

ROTATES = all(checks.values())
print('\n' + ('DETECTION' if ROTATES else
      'UPPER LIMIT -- the data do not require rotation; do not quote V_rot or M_dyn'))

## 5. Results

In [ ]:
import astropy.units as u
from astropy.constants import G

from dingo.utils import cosmo        # Planck18, one cosmology for the whole repo

chain = fit.mcmc_chain                                   # flat, post-burn
names = list(fit.mcmc_param_names)
amp = chain[:, names.index(f'velocity.{amp_key}')]

# Marginalise over the imaging inclination rather than fixing it at the median,
# and widen it by the method-to-method morphology scatter. The pysersic posterior
# alone badly understates this: it gives +-0.09 deg on ID3700, while merely
# allowing a realistic disk thickness (q0 = 0.2) instead of a thin disk moves the
# inclination by 2.65 deg there and 0.71 deg on ID15665 -- and single-Sersic
# model mismatch against a clumpy galaxy is larger still. The chain is
# conditional on one fixed i, so without this V_rot carries a statistical error
# bar where a systematic one dominates.
#
# METHOD_SCATTER is the value the retired fit-kinematics-mcmc.ipynb used. Its
# provenance has not been traced; it is kept because omitting the term entirely
# is the worse error, and it is the right order (2.5 deg) for the q0 effect above.
METHOD_SCATTER = 0.0436                                  # rad, 2.5 deg
INC_LO, INC_HI = 0.0873, 1.4835                          # 5 to 85 deg

rng = np.random.default_rng(0)
inc = rng.choice(np.asarray(sersic['inc']), size=len(amp))
inc = np.clip(inc + rng.normal(0.0, METHOD_SCATTER, len(amp)), INC_LO, INC_HI)
V_sini = amp if amp_key == 'V_sini' else amp*np.sin(INC_RAD)
V_rot  = V_sini/np.sin(inc)

z = float(fit.config['summary']['z'])
# The velocity model works in native NIRCam long-wavelength pixels. 0.063 arcsec
# is the convention diagnostics/run_mcmc.py uses, so an R_v in kpc means the same
# thing whichever path produced it. Do not substitute an oversampled scale here.
KPC_PER_PIX = cosmo.kpc_proper_per_arcmin(z).to(u.kpc/u.arcsec).value*0.063
R_v_kpc = chain[:, names.index('velocity.R_v')]*KPC_PER_PIX

R_KPC = 5.0
M5 = ((V_rot*u.km/u.s)**2 * (R_KPC*u.kpc) / G).to(u.Msun).value
logM5 = np.log10(M5)

def q(x):
    lo, mid, hi = np.percentile(x, [16, 50, 84])
    return mid, mid-lo, hi-mid

for label, arr in (('V sin(i) [km/s]', V_sini), ('V_rot [km/s]', V_rot),
                   ('R_v [kpc]', R_v_kpc), ('log10 M_dyn(<5kpc)', logM5)):
    m, l, h = q(arr)
    print(f'{label:<22} {m:9.3f} -{l:.3f} +{h:.3f}')

np.savez_compressed(out('headline.npz'), z=z, inc_phot=INC_RAD, inc_sigma=INC_SIGMA,
                    Vsini=np.median(V_sini), V_rot=np.median(V_rot),
                    R_v_kpc=np.median(R_v_kpc), logM5=np.median(logM5),
                    inc_method_scatter=METHOD_SCATTER,
                    inc_width_propagated=float(np.std(inc)),
                    rotates=ROTATES, dlnL=null['dlnL'], dBIC=null['dBIC'],
                    start_loss=loss, acceptance=acc, worst_n_over_tau=worst)
print('\nsaved ->', out('headline.npz'))

## 6. Plots

In [ ]:
plot.plot_mcmc_chains(fit.mcmc_sampler.get_chain(), names,
                      nburn=d['n_burn'], filename=out('mcmc_chains.png'))

image_R, image_C, vz_R, vz_C = fit.get_fitting_results()
true_R, true_C = fit.get_true_images()
vel, disp_R, disp_C = fit.get_params()
vel = fitting.with_rotation_amplitude(vel, INC_RAD)

fig, axs = plt.subplots(2, 4, figsize=(18, 9))
plot.plot_kinematics_fitting_result(true_R, image_R, vz_R, image_C, 'R', axs[0], vel, disp_R)
plot.plot_kinematics_fitting_result(true_C, image_C, vz_C, image_R, 'C', axs[1], vel, disp_C)
fig.tight_layout()
fig.savefig(out('fit_summary.png'), dpi=140)